In [ ]:
import json
from pathlib import Path
import pandas as pd

# ==================== 配置 ====================
root_dir = Path("PFM")
# =============================================

data = []

for mil_folder in root_dir.iterdir():
    if not mil_folder.is_dir():
        continue

    method = mil_folder.name

    timestamp_folders = [f for f in mil_folder.iterdir() if f.is_dir() and f.name.startswith('seed')]
    if not timestamp_folders:
        continue
    ts_folder = timestamp_folders[0]
    json_file = ts_folder / "merge_5_fold_metrics.json"
    if not json_file.exists():
        print(f"Warning: {json_file} not found")
        continue

    with open(json_file, encoding="utf-8") as f:
        metrics = json.load(f)

    row = {"method": method}

    for metric_name, values in metrics.items():
        mean = values["mean"]
        std = values["std"]

        mean_str = f"{mean:.4f}"
        std_str = f"{std:.4f}"

        row[metric_name] = f"{mean_str}±{std_str}"

    data.append(row)

df = pd.DataFrame(data)
df = df.sort_values("method").reset_index(drop=True)

output_csv = root_dir / "all_metrics_summary.csv"
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"汇总完成！共处理 {len(data)} 个方法")
print(f"CSV 已保存至: {output_csv}")
print("\n预览：")
print(df)

In [1]:
import os
import pandas as pd
import numpy as np
import ast
import pingouin as pg
from sklearn.metrics import cohen_kappa_score

def analyze_with_pingouin(root_path, hospitals, filename='Ensemble_Weighted_Result.csv'):
    dfs = []
    for hop in hospitals:
        file_path = os.path.join(root_path, hop, filename)
        if not os.path.exists(file_path): continue

        df = pd.read_csv(file_path)

        # --- 核心转换步骤 ---
        def extract_prob(prob_str):
            try:
                # 将字符串 '[0.87, 0.12]' 转为列表 [0.87, 0.12]
                p_list = ast.literal_eval(prob_str)
                # 提取类 1 的概率（假设 index 1 是目标类）
                return p_list[1]
            except:
                return np.nan

        df['base_id'] = df['slide_id'].apply(lambda x: str(x).split('-')[0])

        df[f'probs_{hop}'] = df['probs'].apply(extract_prob)
        df = df.rename(columns={'prediction': f'pred_{hop}'})

        dfs.append(df[['base_id', f'probs_{hop}', f'pred_{hop}', 'label']])

    # 合并数据
    final_df = dfs[0]
    for next_df in dfs[1:]:
        final_df = pd.merge(final_df, next_df, on=['base_id', 'label'], how='inner')

    # 移除任何包含空值的行（转换失败的情况）
    final_df = final_df.dropna()

    # --- 1. 指标波动分析 (Standard Deviation) ---
    prob_cols = [f'probs_{hop}' for hop in hospitals]
    final_df['prob_std'] = final_df[prob_cols].std(axis=1)

    print("-" * 30)
    print(f"## 1. 指标波动分析 (SD)")
    print(f"分析样本数: {len(final_df)}")
    print(f"平均概率标准差 (Mean SD): {final_df['prob_std'].mean():.4f}")
    print(f"SD 越小，说明模型对不同染色中心越不敏感（更稳定）。")

    # --- 2. 一致性分析 (ICC) ---
    # 将宽表转长表
    long_df = final_df.melt(id_vars=['base_id'], value_vars=prob_cols,
                            var_name='hospital', value_name='prob_score')

    # 计算 ICC
    icc = pg.intraclass_corr(data=long_df, targets='base_id', raters='hospital', ratings='prob_score')

    print("\n## 2. 一致性分析 (ICC)")
    # ICC3k: 固定评分者（6家医院）的平均一致性
    target_row = icc[icc['Type'].str.contains(r'\(C,\s*k\)', regex=True)]

    if not target_row.empty:
        val = target_row['ICC'].values[0]
        ci = target_row['CI95'].values[0]
        print(f"多中心一致性 (ICC C,k): {val:.4f}")
        print(f"95% 置信区间: {ci}")
    else:
        # 如果没找到 k 类型，先打印全表查看 Type 列的具体写法
        print("未匹配到特定 ICC 类型，全表结果如下：")
        print(icc[['Type', 'ICC', 'CI95']])

    # --- 3. 分类一致性 (Kappa) ---
    pred_cols = [f'pred_{hop}' for hop in hospitals]
    kappas = []
    for i in range(len(hospitals)):
        for j in range(i + 1, len(hospitals)):
            k = cohen_kappa_score(final_df[pred_cols[i]], final_df[pred_cols[j]])
            kappas.append(k)

    print(f"\n## 3. 分类一致性 (Kappa)")
    print(f"平均 Cohen's Kappa: {np.mean(kappas):.4f}")
    print("-" * 30)

    return final_df

# 运行分析
hospitals = [d for d in os.listdir('Consistent') if os.path.isdir(os.path.join('Consistent', d))]
final_results = analyze_with_pingouin('Consistent', hospitals)
final_results.to_csv('Consistent/result.csv', index=False, encoding="utf-8-sig")

------------------------------
## 1. 指标波动分析 (SD)
分析样本数: 60
平均概率标准差 (Mean SD): 0.0441
SD 越小，说明模型对不同染色中心越不敏感（更稳定）。

## 2. 一致性分析 (ICC)
多中心一致性 (ICC C,k): 0.9939
95% 置信区间: [0.99 1.  ]

## 3. 分类一致性 (Kappa)
平均 Cohen's Kappa: 0.9112
------------------------------


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# ==================== 1. 配置路径与中心列表 ====================
INPUT_FILE = "Consistent/result.csv"
OUTPUT_FILE = "Consistent/table_performance_summary.csv"

# 严格对应你 CSV 中的 6 个中心后缀
centers = ["301", "900", "云南肿瘤", "南京军总", "广西瑞康", "河南安阳"]

# ==================== 2. 读取数据与计算群体共识 ====================
df = pd.read_csv(INPUT_FILE)
pred_cols = [f"pred_{c}" for c in centers]

# 计算每张切片的多数表决（Majority Vote）作为群体共识
majority_vote = df[pred_cols].mode(axis=1)[0].astype(int)
# 计算多中心完全一致率（所有中心对该切片的预测全相同）
unanimous_agreement = df[pred_cols].nunique(axis=1) == 1

# ==================== 3. 核心指标循环统计 ====================
rows = []
for c in centers:
    y_true = df["label"]
    y_pred = df[f"pred_{c}"]
    y_prob = df[f"probs_{c}"]

    # 计算该中心的诊断置信度
    confidence = np.maximum(y_prob, 1 - y_prob)
    # 计算该中心与群体共识的一致率
    agreement_rate = (y_pred == majority_vote).mean()

    rows.append({
        "Staining Center": f"Center {c}",
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Mean Confidence": confidence.mean(),
        "Agreement Rate": agreement_rate,
    })

# ==================== 4. 计算 Overall (多中心集成) 指标 ====================
# 整体平均置信度：所有中心在所有切片上的平均置信度
all_confs = [np.maximum(df[f"probs_{c}"], 1 - df[f"probs_{c}"]) for c in centers]

rows.append({
    "Staining Center": "Overall",
    "Accuracy": accuracy_score(df["label"], majority_vote),
    "Balanced Accuracy": balanced_accuracy_score(df["label"], majority_vote),
    "Macro F1": f1_score(df["label"], majority_vote, average="macro"),
    "Mean Confidence": np.mean(all_confs),
    "Agreement Rate": unanimous_agreement.mean(),  # 整体完全一致率
})

# ==================== 5. 格式化并输出符合模板的 CSV ====================
df_result = pd.DataFrame(rows)

# 保留三位小数，符合标准的学术论文表2/表3规范
metrics = [
    "Accuracy",
    "Balanced Accuracy",
    "Macro F1",
    "Mean Confidence",
    "Agreement Rate",
]
df_result[metrics] = df_result[metrics].round(3)

# 保存文件
df_result.to_csv(OUTPUT_FILE, index=False)

print(f"统计完成！结果已完美保存至: {OUTPUT_FILE}")
print("\n--- 输出 CSV 数据预览 ---")
print(df_result.to_string(index=False))

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, cohen_kappa_score, roc_auc_score

def parse_probs(s):
    return np.array([float(x) for x in re.findall(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|\d+', s.replace('\n', ' '))], dtype=np.float32)

def calc_metrics(y_true, y_pred, y_probs):
    # 自动处理二分类与多分类的 AUC 计算
    is_multiclass = y_probs.shape[1] > 2
    y_true_oh = pd.get_dummies(y_true) if is_multiclass else y_true
    auc_opts = {'multi_class': 'ovr'} if is_multiclass else {}

    macro_auc = roc_auc_score(y_true_oh, y_probs, average='macro', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='macro')
    micro_auc = roc_auc_score(y_true_oh, y_probs, average='micro', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='micro')
    weighted_auc = roc_auc_score(y_true_oh, y_probs, average='weighted', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='weighted')

    return {
        "acc": accuracy_score(y_true, y_pred), "bacc": balanced_accuracy_score(y_true, y_pred),
        "quadratic_kappa": cohen_kappa_score(y_true, y_pred, weights='quadratic'), "linear_kappa": cohen_kappa_score(y_true, y_pred, weights='linear'),
        "macro_f1": f1_score(y_true, y_pred, average='macro'), "macro_pre": precision_score(y_true, y_pred, average='macro', zero_division=0), "macro_recall": recall_score(y_true, y_pred, average='macro'), "macro_auc": macro_auc,
        "micro_f1": f1_score(y_true, y_pred, average='micro'), "micro_pre": precision_score(y_true, y_pred, average='micro', zero_division=0), "micro_recall": recall_score(y_true, y_pred, average='micro'), "micro_auc": micro_auc,
        "weighted_f1": f1_score(y_true, y_pred, average='weighted'), "weighted_pre": precision_score(y_true, y_pred, average='weighted', zero_division=0), "weighted_recall": recall_score(y_true, y_pred, average='weighted'), "weighted_auc": weighted_auc
    }

BASE_DIR = "ensemble_outputs/Random"
mil_folder = Path(BASE_DIR)
seeds = sorted([f.name for f in mil_folder.iterdir() if f.is_dir() and f.name.startswith('seed')])
if not seeds: raise FileNotFoundError("❌ 没找到以seed开头的文件夹！")
seed_folder = seeds[0]

# 1. 收集各 fold 数据及指标
fold_metrics, fold_data = [], []
for fold_idx in range(1, 6):
    df = pd.read_csv(os.path.join(BASE_DIR, seed_folder, f"fold_{fold_idx}", "Infer_Result.csv"))
    df['probs_arr'] = df['probs'].apply(parse_probs)
    fold_data.append(df)
    fold_metrics.append(calc_metrics(df['label'].to_numpy().astype(int), df['prediction'].to_numpy().astype(int), np.stack(df['probs_arr'].values)))

# 2. 算术平均融合概率并保存
combined = pd.concat(fold_data, ignore_index=True)
ensemble_results = []
for slide_id, group in combined.groupby('slide_id'):
    mean_probs = np.mean(np.stack(group['probs_arr'].values), axis=0)
    ensemble_results.append({'slide_id': slide_id, 'label': group['label'].iloc[0], 'probs': f"[{' '.join([f'{x:.7e}' for x in mean_probs])}]", 'prediction': np.argmax(mean_probs)})

pd.DataFrame(ensemble_results)[['slide_id', 'label', 'probs', 'prediction']].to_csv(os.path.join(BASE_DIR, "Ensemble_Weighted_Result.csv"), index=False)

# 3. 组装 16 项指标的 mean 和 std 并保存
final_metrics_json = {key: {"mean": float(np.mean([m[key] for m in fold_metrics])), "std": float(np.std([m[key] for m in fold_metrics]))} for key in fold_metrics[0].keys()}
with open(os.path.join(BASE_DIR, "merge_5_fold_metrics.json"), 'w') as f:
    json.dump(final_metrics_json, f, indent=4)
print("✅ 方案一（16项指标带Std）聚合完成！")


In [ ]:
import os, re, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, cohen_kappa_score, roc_auc_score

def parse_probs(s):
    return np.array([float(x) for x in re.findall(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|\d+', s.replace('\n', ' '))], dtype=np.float32)

def calc_metrics(y_true, y_pred, y_probs):
    is_multiclass = y_probs.shape[1] > 2
    y_true_oh = pd.get_dummies(y_true) if is_multiclass else y_true
    auc_opts = {'multi_class': 'ovr'} if is_multiclass else {}

    macro_auc = roc_auc_score(y_true_oh, y_probs, average='macro', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='macro')
    micro_auc = roc_auc_score(y_true_oh, y_probs, average='micro', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='micro')
    weighted_auc = roc_auc_score(y_true_oh, y_probs, average='weighted', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='weighted')

    return {
        "acc": accuracy_score(y_true, y_pred), "bacc": balanced_accuracy_score(y_true, y_pred),
        "quadratic_kappa": cohen_kappa_score(y_true, y_pred, weights='quadratic'), "linear_kappa": cohen_kappa_score(y_true, y_pred, weights='linear'),
        "macro_f1": f1_score(y_true, y_pred, average='macro'), "macro_pre": precision_score(y_true, y_pred, average='macro', zero_division=0), "macro_recall": recall_score(y_true, y_pred, average='macro'), "macro_auc": macro_auc,
        "micro_f1": f1_score(y_true, y_pred, average='micro'), "micro_pre": precision_score(y_true, y_pred, average='micro', zero_division=0), "micro_recall": recall_score(y_true, y_pred, average='micro'), "micro_auc": micro_auc,
        "weighted_f1": f1_score(y_true, y_pred, average='weighted'), "weighted_pre": precision_score(y_true, y_pred, average='weighted', zero_division=0), "weighted_recall": recall_score(y_true, y_pred, average='weighted'), "weighted_auc": weighted_auc
    }

BASE_DIR = "ensembles/Ext/Top5"
models = [f for f in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, f))]

# 自动匹配第一个子模型文件夹里的第一个seed开头的文件夹
seed_folder = None
for m in models:
    seeds = sorted([f.name for f in Path(os.path.join(BASE_DIR, m)).iterdir() if f.is_dir() and f.name.startswith('seed')])
    if seeds: seed_folder = seeds[0]; break
if not seed_folder: raise FileNotFoundError("❌ 没找到任何以seed开头的文件夹！")

combo_metrics, slide_dict = [], {}
for m in models:
    for fold in range(1, 6):
        p = os.path.join(BASE_DIR, m, seed_folder, f"fold_{fold}", "Infer_Result.csv")
        if not os.path.exists(p): continue
        df = pd.read_csv(p)
        df['probs_arr'] = df['probs'].apply(parse_probs)

        combo_metrics.append(calc_metrics(df['label'].to_numpy().astype(int), df['prediction'].to_numpy().astype(int), np.stack(df['probs_arr'].values)))
        for _, r in df.iterrows():
            slide_dict.setdefault(r['slide_id'], {'label': int(r['label']), 'probs': []})['probs'].append(parse_probs(r['probs']))

# 跨模型多折全局均值融合
ensemble_results = []
for sid, data in slide_dict.items():
    mean_probs = np.mean(np.stack(data['probs']), axis=0)
    ensemble_results.append({'slide_id': sid, 'label': data['label'], 'probs': f"[{' '.join([f'{x:.7e}' for x in mean_probs])}]", 'prediction': np.argmax(mean_probs)})

pd.DataFrame(ensemble_results)[['slide_id', 'label', 'probs', 'prediction']].to_csv(os.path.join(BASE_DIR, "Ensemble_Weighted_Result.csv"), index=False)

# 组装 16 项指标的 mean 和 std 并保存
final_metrics_json = {key: {"mean": float(np.mean([m[key] for m in combo_metrics])), "std": float(np.std([m[key] for m in combo_metrics]))} for key in combo_metrics[0].keys()}
with open(os.path.join(BASE_DIR, "merge_5_fold_metrics.json"), 'w') as f:
    json.dump(final_metrics_json, f, indent=4)
print("✅ 方案二全模型（16项指标带Std）聚合完成！")

In [ ]:
import json
import ast
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    f1_score, precision_score, recall_score, roc_auc_score
)

def calc_metrics(y_true, y_pred, y_probs):
    is_multiclass = y_probs.shape[1] > 2
    y_true_oh = pd.get_dummies(y_true) if is_multiclass else y_true
    auc_opts = {'multi_class': 'ovr'} if is_multiclass else {}

    macro_auc = roc_auc_score(y_true_oh, y_probs, average='macro', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='macro')
    micro_auc = roc_auc_score(y_true_oh, y_probs, average='micro', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='micro')
    weighted_auc = roc_auc_score(y_true_oh, y_probs, average='weighted', **auc_opts) if is_multiclass else roc_auc_score(y_true, y_probs[:, 1], average='weighted')

    return {
        "acc": accuracy_score(y_true, y_pred), "bacc": balanced_accuracy_score(y_true, y_pred),
        "quadratic_kappa": cohen_kappa_score(y_true, y_pred, weights='quadratic'), "linear_kappa": cohen_kappa_score(y_true, y_pred, weights='linear'),
        "macro_f1": f1_score(y_true, y_pred, average='macro'), "macro_pre": precision_score(y_true, y_pred, average='macro', zero_division=0), "macro_recall": recall_score(y_true, y_pred, average='macro'), "macro_auc": macro_auc,
        "micro_f1": f1_score(y_true, y_pred, average='micro'), "micro_pre": precision_score(y_true, y_pred, average='micro', zero_division=0), "micro_recall": recall_score(y_true, y_pred, average='micro'), "micro_auc": micro_auc,
        "weighted_f1": f1_score(y_true, y_pred, average='weighted'), "weighted_pre": precision_score(y_true, y_pred, average='weighted', zero_division=0), "weighted_recall": recall_score(y_true, y_pred, average='weighted'), "weighted_auc": weighted_auc
    }

# 1. 读取 CSV
csv_path = 'ensembles/SPE/seed/Ensemble_Weighted_Result.csv'
df = pd.read_csv(csv_path)

# 2. 解析 probs 字符串为 numpy 数组
y_true = df['label'].values.astype(int)
y_pred = df['prediction'].values.astype(int)
probs = np.array([ast.literal_eval(p) for p in df['probs']])

# 3. 100 次 Bootstrap
np.random.seed(42)
n_samples = len(df)
n_bootstrap = 100

# 初始化存储
all_results = {key: [] for key in calc_metrics(y_true, y_pred, probs).keys()}

for _ in range(n_bootstrap):
    idx = np.random.choice(n_samples, size=n_samples, replace=True)
    metrics = calc_metrics(y_true[idx], y_pred[idx], probs[idx])
    for k, v in metrics.items():
        all_results[k].append(v)

# 4. 计算 mean 和 std，保存为嵌套字典格式
result_json = {}
for k, vals in all_results.items():
    result_json[k] = {
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals))
    }

# 5. 保存到 result.json
with open('ensembles/SPE/seed/merge_5_fold_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(result_json, f, ensure_ascii=False, indent=2)

print("Bootstrap 完成，结果已保存至 result.json")
print(json.dumps(result_json, ensure_ascii=False, indent=2))

In [ ]:
from utils.general_utils import merge_k_fold_logs
merge_k_fold_logs('/NAS2/Data1/lbliao/Code-195/MIL_BASELINE/result/Contrast/Consistent/云南肿瘤','123')